[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C67_LLM_Judge_Course/05_reward_models/05_reward_models.ipynb)

# 05 · 奖励模型与过优化（BT 损失训练 RM / KL–奖励曲线 / hack 形态 / 集成 / 切片评测）

目标：把「judge 变成训练信号之后会发生什么」从一句警告，变成**能画出来、能定位拐点的曲线**。

本 notebook 你会亲手实现：
1. **用 BT 损失训练一个奖励模型** —— 与模块 04 的 BT 是同一个损失，只是参数化不同
2. **过优化的倒 U 曲线** —— best-of-n 与 RL 两条路径，代理奖励涨而真实质量跌
3. **拐点定位** —— 拟合 $R(d) = d(\alpha - \beta d)$，解出 $d^* = \alpha/2\beta$
4. **分布外退化** —— RM 在训练分布内很准，在被优化推到的区域完全失准
5. **奖励集成与保守化** —— 取最小值 / 均值减方差，把拐点推后多少
6. **RewardBench 风格的切片评测** —— 为什么总准确率没有信息量
7. **独立信号** —— 用一个与 RM 误差不相关的验证量，把拐点真的抓出来

> 心智模型：**RM 只在训练分布上被约束，而优化过程恰恰会把策略推到分布之外。
> 过优化不是实现缺陷，是数学上必然的——你只能推后它、检测它，不能消除它。**

## 1 · 用 BT 损失训练一个奖励模型

为了能在 CPU 上跑通并且**知道真值**，我们用一个线性 RM 和一个可控的"回答空间"：
每个回答由若干个特征描述（其中一部分是真实质量维度，一部分是"表面属性"如长度）。

真实效用只看质量维度；但**偏好数据是由一个带长度偏好的标注者生成的**——
于是 RM 会学到「长 = 好」，这正是模块 02 的长度偏差在训练侧的样子。

In [ ]:
import math, json, itertools
from collections import Counter, defaultdict
import numpy as np

D_QUALITY, D_SURFACE = 4, 2          # 4 个质量维度 + 2 个表面维度（长度/格式）
D = D_QUALITY + D_SURFACE

W_QUALITY = np.array([1.0, 0.8, 0.6, 0.4])       # 质量维度的真实权重
KAPPA = np.array([0.45, 0.30])                   # 表面维度过头之后的惩罚
W_LABELER = np.concatenate([W_QUALITY, [1.0, 0.5]])   # 标注者还偏好长/格式（线性、无上限）

def true_utility(Y):
    # 真实效用有两个现实特征，而代理奖励两个都没有：
    #   ① 质量维度**会饱和**（tanh）——再好也好不到哪去；
    #   ② 表面维度**过头会扣分**（-kappa*x^2）——适度的长度/结构有用，过头则是注水。
    # 代理奖励是线性无上限的，这个失配就是过优化的全部来源。
    Y = np.asarray(Y, dtype=float)
    return np.tanh(Y[:, :D_QUALITY]) @ W_QUALITY - (Y[:, D_QUALITY:] ** 2) @ KAPPA

def labeler_score(Y):
    # 标注者的打分：线性、无饱和、无惩罚——他们在单条比较里看不出「过头」
    return np.asarray(Y, dtype=float) @ W_LABELER

def sample_responses(n, rng, scale=1.0):
    """从参考策略采样回答：每个回答是一个 D 维特征向量。"""
    return rng.normal(0, scale, size=(n, D))

def make_preference_data(n_pairs, rng, noise=0.6):
    """标注者按 labeler_score 的 BT 概率给出偏好（带噪声）。"""
    A = sample_responses(n_pairs, rng)
    B = sample_responses(n_pairs, rng)
    d = (labeler_score(A) - labeler_score(B)) / noise
    win_a = (rng.random(n_pairs) < 1 / (1 + np.exp(-np.clip(d, -30, 30)))).astype(int)
    return A, B, win_a

def train_rm(A, B, win_a, l2=1e-3, lr=0.5, iters=4000):
    """线性 RM，BT 损失（与模块 04 的 fit_bt 是同一个损失）。"""
    A, B = np.asarray(A, float), np.asarray(B, float)
    y = np.asarray(win_a, float)
    w = np.zeros(A.shape[1])
    for _ in range(iters):
        z = (A - B) @ w
        p = 1 / (1 + np.exp(-np.clip(z, -30, 30)))
        grad = (A - B).T @ (p - y) / len(y) + l2 * w
        w -= lr * grad
    return w

rng = np.random.default_rng(0)
A_tr, B_tr, y_tr = make_preference_data(20000, rng)
w_rm = train_rm(A_tr, B_tr, y_tr)

print(f"{'维度':<14}{'标注者 W':>12}{'学到的 RM':>12}{'真实效用的形状':>20}")
names = [f'quality_{i}' for i in range(D_QUALITY)] + ['length', 'format']
shapes = [f'{W_QUALITY[i]:.1f}·tanh(x)' for i in range(D_QUALITY)] +          [f'-{KAPPA[0]:.2f}x²', f'-{KAPPA[1]:.2f}x²']
for i, nm in enumerate(names):
    print(f'{nm:<14}{W_LABELER[i]:>12.2f}{w_rm[i]:>12.2f}{shapes[i]:>20}')

assert w_rm[D_QUALITY] > 0.5, 'RM 必然学到了标注者的长度偏好'
assert np.corrcoef(w_rm[:D_QUALITY], W_QUALITY)[0, 1] > 0.9
print('\n✅ RM 学到的质量维度权重与真值排序一致——但它**同时学到了长度偏好**，')
print('   而且是**线性无上限**的：真实效用里长度过头会扣分，RM 里长度永远加分。')
print('   这不是 bug：RM 忠实地拟合了偏好数据，而偏好数据本身就带着标注者的偏差。')
print('   → 模块 02 的 judge 偏差，在训练侧就变成了 RM 权重里的一项。')

In [ ]:
# RM 的静态准确率看起来很好——这正是问题所在
A_te, B_te, y_te = make_preference_data(5000, np.random.default_rng(1))
pred = ((A_te - B_te) @ w_rm > 0).astype(int)
acc_labeler = float((pred == y_te).mean())
true_pref = (true_utility(A_te) > true_utility(B_te)).astype(int)
acc_true = float((pred == true_pref).mean())
print(f'RM 与**标注者**的一致率:  {acc_labeler:.1%}   ← 静态基准测的是这个')
print(f'RM 与**真实效用**的一致率: {acc_true:.1%}   ← 我们真正关心的是这个')
assert acc_labeler > acc_true
print(f'\n差距 {acc_labeler - acc_true:.1%} —— 这部分完全来自标注者的长度/格式偏好。')
print('✅ 静态基准准确率高，只说明 RM 忠实地复制了标注者（包括他们的偏差）。')

## 2 · 过优化的倒 U 曲线：best-of-n

In [ ]:
def best_of_n(n, n_prompts, rng, w_score):
    """对每个 prompt 采 n 个候选，按 w_score 选最好的一个。
    返回 (被选中回答的代理奖励均值, 真实效用均值, KL 估计)。"""
    proxy, true_u = [], []
    for _ in range(n_prompts):
        cands = sample_responses(n, rng)
        s = cands @ w_score
        k = int(np.argmax(s))
        proxy.append(float(s[k]))
        true_u.append(float(true_utility(cands[k:k+1])[0]))
    kl = math.log(n) - (n - 1) / n if n > 1 else 0.0
    return float(np.mean(proxy)), float(np.mean(true_u)), kl

rng = np.random.default_rng(7)
NS = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
rows = []
for n in NS:
    pr, tu, kl = best_of_n(n, 3000, rng, w_rm)
    rows.append((n, kl, pr, tu))

base_true = rows[0][3]
print(f"{'n':>6}{'KL':>8}{'代理奖励(RM)':>14}{'真实效用':>12}{'相对 n=1 的提升':>16}")
for n, kl, pr, tu in rows:
    print(f'{n:>6}{kl:>8.2f}{pr:>14.3f}{tu:>12.3f}{tu - base_true:>+16.3f}')

proxy_seq = [r[2] for r in rows]
true_seq = [r[3] for r in rows]
assert all(proxy_seq[i] < proxy_seq[i+1] for i in range(len(proxy_seq)-1)), '代理奖励必然单调上升'
peak = int(np.argmax(true_seq))
assert 0 < peak < len(true_seq) - 1, '真实效用必须是倒 U（先升后降）'
print(f'\n✅ 代理奖励单调上升到底；真实效用在 n={NS[peak]} 达到峰值，之后开始**下降**。')
print(f'   n 从 {NS[peak]} 加到 {NS[-1]}：代理奖励 +{proxy_seq[-1]-proxy_seq[peak]:.2f}，'
      f'真实效用 {true_seq[-1]-true_seq[peak]:+.2f}')
print('   **而在真实项目里，你只能看到上面那条线。**')

In [ ]:
# 被选中的回答，长什么样？——第 3 节 hack 形态的定量版本
def selected_profile(n, n_prompts, rng, w_score):
    feats = []
    for _ in range(n_prompts):
        c = sample_responses(n, rng)
        feats.append(c[int(np.argmax(c @ w_score))])
    return np.mean(feats, axis=0)

print(f"{'n':>6}" + ''.join(f'{nm:>12}' for nm in names))
for n in [1, 8, 64, 512]:
    prof = selected_profile(n, 2000, np.random.default_rng(11), w_rm)
    print(f'{n:>6}' + ''.join(f'{v:>12.3f}' for v in prof))

p1 = selected_profile(1, 2000, np.random.default_rng(11), w_rm)
p512 = selected_profile(512, 2000, np.random.default_rng(11), w_rm)
len_growth = p512[D_QUALITY] - p1[D_QUALITY]
qual_growth = p512[0] - p1[0]
print(f'\nn 从 1 到 512：length 维度 +{len_growth:.2f}，最重要的质量维度 +{qual_growth:.2f}')
assert len_growth > 0.5
print('✅ 优化压力把「长度」这个维度推得很高——**这就是长度膨胀的机制**。')
print('   模型没有在「钻空子」，它只是在沿着 RM 给的梯度走。')

## 3 · 拐点定位：拟合 $R(d) = d(\alpha - \beta d)$

In [ ]:
def fit_overopt_curve(d_list, gain_list):
    """拟合 R(d) = alpha*d - beta*d^2（无截距的二次），返回 (alpha, beta, d_star)。"""
    d = np.asarray(d_list, dtype=float)
    g = np.asarray(gain_list, dtype=float)
    X = np.column_stack([d, -d ** 2])            # 设计矩阵，无截距
    coef, *_ = np.linalg.lstsq(X, g, rcond=None)
    alpha, beta = float(coef[0]), float(coef[1])
    d_star = alpha / (2 * beta) if beta > 0 else float('inf')
    return alpha, beta, d_star

d_vals = np.array([math.sqrt(r[1]) for r in rows])
gains = np.array([r[3] - base_true for r in rows])
alpha, beta, d_star = fit_overopt_curve(d_vals, gains)
print(f'拟合结果: alpha = {alpha:.3f}, beta = {beta:.3f}')
print(f'拐点 d* = alpha/(2*beta) = {d_star:.3f}  →  KL* = {d_star**2:.3f}')
n_star = math.exp(d_star ** 2 + 1) if d_star < 10 else float('inf')
print(f'对应的 best-of-n 大致在 n ≈ {n_star:.0f}')
assert beta > 0, '必须有正的二次惩罚项，否则不存在拐点'
assert 0 < d_star < d_vals.max(), '拐点应当落在实验范围内'
observed_peak_d = d_vals[int(np.argmax(gains))]
print(f'实测峰值出现在 d = {observed_peak_d:.3f}，拟合拐点 d* = {d_star:.3f}')
assert abs(d_star - observed_peak_d) < 0.8
print('\n✅ 拟合曲线定位到的拐点与实测峰值一致。')
print('   实践建议：**把生产用的 KL 预算设在 d* 之前留有余量的位置**——')
print('   d* 本身有估计误差，而越过它的代价是不对称的（真实质量下降，且你看不见）。')

## 4 · 分布外退化：RM 在被推到的区域有多不准

In [ ]:
def rm_agreement_at_optimization_level(n, n_pairs, rng, w_rm):
    """在 best-of-n 优化后的分布上，测 RM 与真实效用的一致率。"""
    agree = 0
    for _ in range(n_pairs):
        # 两个都是被 best-of-n 选出来的回答（即优化后分布）
        c1 = sample_responses(n, rng); a = c1[int(np.argmax(c1 @ w_rm))]
        c2 = sample_responses(n, rng); b = c2[int(np.argmax(c2 @ w_rm))]
        if (a @ w_rm > b @ w_rm) == (true_utility(a[None])[0] > true_utility(b[None])[0]):
            agree += 1
    return agree / n_pairs

print(f"{'优化强度 n':>12}{'RM 与真实效用的一致率':>24}")
for n in [1, 8, 64, 512]:
    a_ = rm_agreement_at_optimization_level(n, 2000, np.random.default_rng(13), w_rm)
    print(f'{n:>12}{a_:>24.1%}')

a1 = rm_agreement_at_optimization_level(1, 2000, np.random.default_rng(13), w_rm)
a512 = rm_agreement_at_optimization_level(512, 2000, np.random.default_rng(13), w_rm)
assert a512 < a1
print(f'\n✅ 一致率从 {a1:.1%} 掉到 {a512:.1%}——**RM 在它自己推出来的分布上变得更不准了**。')
print('   机制：优化把样本推到「长度维度很高」的区域，而在那个区域，')
print('   RM 的分数几乎全部由长度决定，与真实质量的关联被稀释了。')
print('   → 这就是为什么 RewardBench 类的静态准确率是必要条件而非充分条件。')

## 5 · 集成与保守化：把拐点推后多少

In [ ]:
def train_rm_ensemble(K, n_pairs, base_seed=100, l2=1e-3):
    """K 个 RM，各自用不同的数据划分与初始化（这里用不同 seed 的数据）。"""
    ws = []
    for k in range(K):
        rg = np.random.default_rng(base_seed + k)
        A, B, y = make_preference_data(n_pairs, rg)
        ws.append(train_rm(A, B, y, l2=l2))
    return np.array(ws)

W_ENS = train_rm_ensemble(5, 8000)
print('集成中各 RM 的 length 权重:', np.round(W_ENS[:, D_QUALITY], 3))
print('→ 方向完全一致（都学到了长度偏好），因为**偏差来自数据而不是随机性**')

def bon_with_scorer(n, n_prompts, rng, score_fn):
    proxy, true_u = [], []
    for _ in range(n_prompts):
        c = sample_responses(n, rng)
        s = score_fn(c)
        k = int(np.argmax(s))
        proxy.append(float(s[k])); true_u.append(float(true_utility(c[k:k+1])[0]))
    return float(np.mean(proxy)), float(np.mean(true_u))

SCORERS = {
    '单个 RM':        lambda c: c @ w_rm,
    '集成均值':       lambda c: (c @ W_ENS.T).mean(axis=1),
    '集成最小值':     lambda c: (c @ W_ENS.T).min(axis=1),
    '保守化 μ-λσ':    lambda c: (c @ W_ENS.T).mean(axis=1) - 1.0 * (c @ W_ENS.T).std(axis=1),
}
print(f"\n{'策略':<16}" + ''.join(f'{f"n={n}":>10}' for n in [1, 16, 128, 512]))
results = {}
for name, fn in SCORERS.items():
    tus = []
    for n in [1, 16, 128, 512]:
        _, tu = bon_with_scorer(n, 2000, np.random.default_rng(17), fn)
        tus.append(tu)
    results[name] = tus
    print(f'{name:<16}' + ''.join(f'{v:>10.3f}' for v in tus))

single_drop = results['单个 RM'][1] - results['单个 RM'][-1]
cons_drop = results['保守化 μ-λσ'][1] - results['保守化 μ-λσ'][-1]
print(f'\n从峰值到 n=512 的真实效用跌幅: 单个 RM {single_drop:.3f} | 保守化 {cons_drop:.3f}')
assert cons_drop < single_drop, '保守化应当减缓过优化'
print('✅ 保守化（μ-λσ）确实减缓了跌幅——它把「不确定的高分」自动折价了。')
print('⚠️ 但注意第一行的观察：**所有 RM 的 length 权重方向一致**，')
print('   所以集成治不了长度偏差这类「共有偏差」——它只能治各 RM 独有的那部分误差。')
print('   （与模块 02 第 7 节「集成只能治噪声，治不了共有偏差」是同一条结论。）')

## 6 · RewardBench 风格的切片评测：总准确率没有信息量

In [ ]:
def make_slice(kind, n, rng):
    """造不同难度/类型的偏好对。返回 (A, B, 真实偏好标签)。"""
    A = sample_responses(n, rng)
    if kind == 'easy':
        B = A - np.column_stack([rng.uniform(0.8, 1.5, n)] + [np.zeros(n)] * (D - 1))
    elif kind == 'hard':
        B = A.copy()
        B[:, 0] -= rng.uniform(0.05, 0.15, n)          # 质量只差一点点
    elif kind == 'style_trap':
        B = A.copy()
        B[:, 0] -= rng.uniform(0.1, 0.3, n)            # B 质量略差
        B[:, D_QUALITY] += rng.uniform(1.0, 2.0, n)    # 但 B 更长 —— 陷阱
    else:
        raise ValueError(kind)
    label = (true_utility(A) > true_utility(B)).astype(int)
    return A, B, label

print(f"{'切片':<14}{'单个 RM':>12}{'集成最小值':>14}{'样本数':>8}")
accs = {}
for kind in ['easy', 'hard', 'style_trap']:
    A_, B_, lab = make_slice(kind, 4000, np.random.default_rng(23))
    a_single = float((((A_ - B_) @ w_rm > 0).astype(int) == lab).mean())
    ens_s = ((A_ @ W_ENS.T).min(axis=1) - (B_ @ W_ENS.T).min(axis=1) > 0).astype(int)
    a_ens = float((ens_s == lab).mean())
    accs[kind] = (a_single, a_ens)
    print(f'{kind:<14}{a_single:>12.1%}{a_ens:>14.1%}{4000:>8}')

overall = np.mean([accs[k][0] for k in accs])
print(f'\n"总准确率"（三片平均）: {overall:.1%}')
assert accs['easy'][0] > 0.95, 'easy 片必然饱和'
assert accs['style_trap'][0] < 0.5, '风格陷阱片上 RM 应当比瞎猜还差'
print(f'\n✅ 三个数字讲了完全不同的故事：')
print(f'   easy {accs["easy"][0]:.0%}（饱和，无信息）· '
      f'hard {accs["hard"][0]:.0%} · style_trap {accs["style_trap"][0]:.0%}')
print('   **风格陷阱片上比瞎猜还差**——因为 RM 学到的长度偏好在这里直接指向错误答案。')
print('   而这三片的平均值把这个致命弱点完全掩盖了。→ 切片报告，不报总分。')

## 7 · 独立信号：把拐点真的抓出来

In [ ]:
def verifiable_signal(n, n_prompts, rng, w_score, q_thresh=0.6, len_budget=1.5):
    """一个「可验证任务正确率」的代理，模拟一条真实的自动检查：
    「答案正确（质量维度够高）**且** 输出在长度预算之内」。
    后半条正是 RM 完全没有的约束——所以这个信号与 RM 的误差项不相关。
    这才是「独立」的技术含义，而不是「换一个更强的模型来看」。"""
    ok = 0
    for _ in range(n_prompts):
        c = sample_responses(n, rng)
        best = c[int(np.argmax(c @ w_score))]
        ok += int(best[0] > q_thresh and abs(best[D_QUALITY]) < len_budget)
    return ok / n_prompts

def correlated_signal(n, n_prompts, rng, w_score):
    """一个"不独立"的验证信号：用同门 RM（同样的数据分布训出来）打分。"""
    w_sibling = W_ENS[0]
    vals = []
    for _ in range(n_prompts):
        c = sample_responses(n, rng)
        best = c[int(np.argmax(c @ w_score))]
        vals.append(float(best @ w_sibling))
    return float(np.mean(vals))

print(f"{'n':>6}{'代理奖励(RM)':>14}{'同门 RM 验证':>14}{'可验证信号':>12}{'真实效用':>12}")
ind_seq, corr_seq = [], []
for n in [1, 8, 32, 128, 512]:
    pr, tu, _ = best_of_n(n, 1500, np.random.default_rng(31), w_rm)
    ind = verifiable_signal(n, 1500, np.random.default_rng(31), w_rm)
    cor = correlated_signal(n, 1500, np.random.default_rng(31), w_rm)
    ind_seq.append(ind); corr_seq.append(cor)
    print(f'{n:>6}{pr:>14.3f}{cor:>14.3f}{ind:>12.1%}{tu:>12.3f}')

assert corr_seq[-1] > corr_seq[0], '同门 RM 的验证信号会一路上升——在过优化区域依然如此'
peak_ind = int(np.argmax(ind_seq))
assert peak_ind < len(ind_seq) - 1, '独立信号必须能看到下降'
print(f'\n✅ 同门 RM 的验证值一路上升（{corr_seq[0]:.2f} → {corr_seq[-1]:.2f}），')
print('   它在你最需要它的时候恰好失效——因为它与被优化的 RM 共享同样的误差项。')
print(f'   而独立的可验证信号在 n={[1,8,32,128,512][peak_ind]} 见顶后开始下降，成功抓到了拐点。')
print('\n   → 便宜且强推荐的组合：**可验证任务正确率 + 行为分布监控**，')
print('     两者全自动、成本近乎为零，覆盖「能力退化」与「形态漂移」两类表现。')

## ✏️ 练习 1：best-of-n 的 KL 与拐点换算

实现 `bon_kl(n)`（$\log n - \frac{n-1}{n}$）与
`kl_to_n(kl)`（给定 KL 预算，反解最大可用的 $n$，返回满足 `bon_kl(n) <= kl` 的最大整数 $n \ge 1$）。

In [ ]:
def bon_kl(n):
    # TODO
    raise NotImplementedError

def kl_to_n(kl, n_max=100000):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert bon_kl(1) == 0.0
assert abs(bon_kl(2) - (math.log(2) - 0.5)) < 1e-12
assert kl_to_n(0.0) == 1
n_ = kl_to_n(2.0)
assert bon_kl(n_) <= 2.0 < bon_kl(n_ + 1)
print(f"{'n':>8}{'KL':>10}    |    {'KL 预算':>10}{'最大 n':>10}")
for n, kl in zip([1, 4, 16, 64, 256], [0.5, 1.0, 2.0, 3.0, 4.0]):
    print(f'{n:>8}{bon_kl(n):>10.3f}    |    {kl:>10.1f}{kl_to_n(kl):>10}')
print(f'\n本 notebook 拟合出的 KL* = {d_star**2:.2f} → 对应 n ≈ {kl_to_n(d_star**2)}')
print('✅ 练习 1 通过：KL 预算与 best-of-n 的 n 可以互相换算——')
print('   这让「RL 里该设多大的 KL 惩罚」和「best-of-n 该取多少」变成同一个决策。')

## ✏️ 练习 2：保守化系数 λ 的选择

实现 `sweep_lambda(lams, n, n_prompts=1500, seed=0)`：对每个 λ，
用 `μ - λσ` 作为打分函数跑 best-of-n，返回 `[(λ, 真实效用), ...]`。
用它找出使真实效用最大的 λ。

In [ ]:
def sweep_lambda(lams, n, n_prompts=1500, seed=0):
    # TODO：复用 bon_with_scorer 与 W_ENS
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
LAMS = [0.0, 0.5, 1.0, 2.0, 4.0]
res_l = sweep_lambda(LAMS, n=256, seed=41)
assert len(res_l) == len(LAMS)
best_lam = max(res_l, key=lambda t: t[1])
print(f"{'lambda':>8}{'真实效用':>12}")
for l_, u_ in res_l:
    mark = '  ← 最优' if l_ == best_lam[0] else ''
    print(f'{l_:>8.1f}{u_:>12.3f}{mark}')
assert best_lam[0] > 0.0, 'λ=0（不保守化）不应当是最优'
u0 = dict(res_l)[0.0]
assert best_lam[1] > u0
print(f'\n最优 λ = {best_lam[0]}，相对 λ=0 提升真实效用 {best_lam[1]-u0:+.3f}')
print('✅ 练习 2 通过：λ 太小起不到保守化作用，太大则把有效信号也压掉了——')
print('   这条曲线要用**独立信号**扫，不能用 RM 自己扫（那样 λ=0 永远最优）。')

## ✏️ 练习 3：hack 形态的自动检测

实现 `hack_monitor(profile_base, profile_now, names, thresh=0.5)`：
比较优化前后被选中回答的平均特征，返回所有漂移超过 `thresh` 的维度
`[(维度名, 漂移量), ...]`，按漂移量降序。

In [ ]:
def hack_monitor(profile_base, profile_now, names, thresh=0.5):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
alerts = hack_monitor(p1, p512, names, thresh=0.5)
print('优化前后的特征漂移告警:')
for nm, delta in alerts:
    print(f'  {nm:<12} {delta:+.3f}')
assert any(nm == 'length' for nm, _ in alerts), 'length 维度必须被告警'
assert alerts == sorted(alerts, key=lambda t: -abs(t[1])), '必须按漂移量降序'
assert hack_monitor(p1, p1, names) == [], '没有漂移时不应告警'
print('\n✅ 练习 3 通过：这就是「行为分布监控」的最小实现——')
print('   它不测质量，但能告诉你「模型正在往哪个方向被 hack」，而且成本近乎为零。')

## ✏️ 练习 4：切片评测卡

实现 `rm_eval_card(w, slices, n=3000, seed=0)`：对每个切片算准确率，
返回 `{'per_slice': {切片: 准确率}, 'macro': 宏平均, 'worst_slice': (名字, 准确率)}`。
**`worst_slice` 才是决定 RM 能不能用的那个数。**

In [ ]:
def rm_eval_card(w, slices, n=3000, seed=0):
    # TODO：用 make_slice 造数据
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
card = rm_eval_card(w_rm, ['easy', 'hard', 'style_trap'], seed=53)
assert set(card) == {'per_slice', 'macro', 'worst_slice'}
assert card['worst_slice'][0] == 'style_trap'
assert card['worst_slice'][1] < card['macro']
for k, v in card['per_slice'].items():
    print(f'  {k:<14} {v:.1%}')
print(f"  {'macro':<14} {card['macro']:.1%}")
print(f"  {'worst':<14} {card['worst_slice'][0]} @ {card['worst_slice'][1]:.1%}")
card_ens = rm_eval_card(W_ENS.min(axis=0), ['easy', 'hard', 'style_trap'], seed=53)
print(f"\n集成最小值的 worst slice: {card_ens['worst_slice'][1]:.1%}")
print('✅ 练习 4 通过：**报 worst_slice，不报 macro**——')
print('   一个在风格陷阱片上低于 50% 的 RM，无论总分多高都不该被用作训练信号。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def bon_kl(n):
    n = int(n)
    return 0.0 if n <= 1 else math.log(n) - (n - 1) / n

def kl_to_n(kl, n_max=100000):
    best = 1
    for n in range(1, n_max + 1):
        if bon_kl(n) <= kl:
            best = n
        else:
            break
    return best

In [ ]:
# 练习 2 参考答案
def sweep_lambda(lams, n, n_prompts=1500, seed=0):
    out = []
    for lam in lams:
        fn = (lambda c, L=lam: (c @ W_ENS.T).mean(axis=1) - L * (c @ W_ENS.T).std(axis=1))
        _, tu = bon_with_scorer(n, n_prompts, np.random.default_rng(seed), fn)
        out.append((lam, tu))
    return out

In [ ]:
# 练习 3 参考答案
def hack_monitor(profile_base, profile_now, names, thresh=0.5):
    base = np.asarray(profile_base, dtype=float)
    now = np.asarray(profile_now, dtype=float)
    alerts = [(names[i], float(now[i] - base[i])) for i in range(len(names))
              if abs(now[i] - base[i]) > thresh]
    return sorted(alerts, key=lambda t: -abs(t[1]))

In [ ]:
# 练习 4 参考答案
def rm_eval_card(w, slices, n=3000, seed=0):
    per = {}
    for i, kind in enumerate(slices):
        A_, B_, lab = make_slice(kind, n, np.random.default_rng(seed + i))
        per[kind] = float((((A_ - B_) @ np.asarray(w) > 0).astype(int) == lab).mean())
    worst = min(per.items(), key=lambda t: t[1])
    return {'per_slice': per,
            'macro': float(np.mean(list(per.values()))),
            'worst_slice': worst}

---
## 🧪 真实工程胶囊：RLHF 流程里必须接上的四条线

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════
# A. 训练 RM 的最小骨架（BT 损失 —— 与模块 04 是同一个损失）
# ══════════════════════════════════════════════════════════════════
import torch, torch.nn.functional as F
def rm_loss(model, batch):
    r_chosen  = model(batch["prompt"], batch["chosen"]).squeeze(-1)
    r_rejected = model(batch["prompt"], batch["rejected"]).squeeze(-1)
    return -F.logsigmoid(r_chosen - r_rejected).mean()
# 常见附加项：
#   + 0.01 * (r_chosen**2 + r_rejected**2).mean()   # 防止分数尺度漂移
#   + margin loss（当偏好标注带强度时）

# ══════════════════════════════════════════════════════════════════
# B. 训练时必须并行跑的四条监控线（缺一条你就是在盲飞）
# ══════════════════════════════════════════════════════════════════
MONITORS = {
  # 1. 代理奖励 —— 你唯一"免费"的信号，但它在过优化时依然上升
  "proxy_reward":    lambda ckpt: eval_rm_score(ckpt),
  # 2. 可验证任务正确率 —— 独立、自动、便宜。**最重要的一条**
  "verifiable_acc":  lambda ckpt: run_math_and_code_tests(ckpt),
  # 3. 行为分布 —— 不测质量，但能看出在往哪个方向被 hack
  "behavior":        lambda ckpt: {
      "mean_output_tokens": ...,      # 长度膨胀
      "md_elements_per_1k": ...,      # 格式套路化
      "refusal_rate":       ...,      # 拒答漂移（双向）
      "hedge_word_rate":    ...,      # 自信化（含糊表述消失）
      "ece":                ...,      # 校准恶化（模块 03）
  },
  # 4. 人类抽样 —— 金标准。每轮 200 条也远好过没有
  "human_sample":    lambda ckpt: collect_human_prefs(ckpt, n=200),
}
# 停止规则：verifiable_acc 连续两个检查点下降 → 停，回退到上一个检查点。
# **不要**用 proxy_reward 做停止规则——它在过优化区域依然上升。

# ══════════════════════════════════════════════════════════════════
# C. 硬约束：能验证的绝不交给 RM（第 7 节）
# ══════════════════════════════════════════════════════════════════
def total_reward(prompt, response):
    # 1) 可验证的硬门禁：不合格直接负奖励，不进 RM
    if not schema_valid(response):          return -1.0
    if not tests_pass(prompt, response):    return -1.0
    # 2) 已知 hack 形态的负向验证器（比指望 RM 学会不喜欢它们有效得多）
    penalty = 0.0
    penalty += 0.2 * has_boilerplate_opening(response)
    penalty += 0.2 * (md_element_density(response) > MD_DENSITY_P95)
    # 3) 剩余维度才交给 RM，并做保守化
    scores = np.array([rm(prompt, response) for rm in RM_ENSEMBLE])
    return scores.mean() - LAMBDA * scores.std() - penalty

# ══════════════════════════════════════════════════════════════════
# D. RM 的评测卡（发布 RM 时必须附带）
# ══════════════════════════════════════════════════════════════════
# per-slice accuracy: chat / chat-hard / safety(双向) / reasoning / style-trap
# worst slice        ← **决定能不能用的就是这个数**
# 校准: 分差 1.0 实测对应多少胜率（BT 性质要求 73%）
# 分布外探针: 在被优化过的策略输出上的一致率
# 集成信息: K、各 RM 在关键维度上的权重方向是否一致
'''
print(RECIPE)

### 小结

| 你学到的 | 一句话 | 用在哪 |
|---|---|---|
| RM = 参数化的 BT | 训练目标就是模块 04 的 BT 损失，只是查找表变成了函数 | 理解 RM |
| 过优化必然发生 | RM 只在训练分布上被约束，而优化会把策略推到分布外 | 预期管理 |
| 拐点 $d^*=\alpha/2\beta$ | 拟合独立信号的曲线定位，KL 预算留余量 | 设 KL 惩罚 / 选 n |
| hack 形态 | 每一种都对应模块 02 的一个 judge 偏差——可以事先预测 | 布置监控 |
| 集成的边界 | 治各 RM 独有的误差，治不了偏好数据里的共有偏差 | 选缓解手段 |
| 切片评测 | 报 worst slice，不报 macro；静态准确率是必要不充分条件 | 发布 RM |
| 独立信号 | 与 RM 误差不相关才叫独立；可验证任务 + 行为监控最划算 | RLHF 流程必备 |

**全课收尾**：00 仪器 → 01 设计 → 02 去偏 → 03 元评测 → 04 排名 → 05 训练信号。
一条逻辑贯穿始终：**先验证测量仪器，再相信读数；而一旦仪器变成目标，它就开始失效。**

**下一门课 C68 · Eval 基础设施与线上监控**：把这一切工程化——
judge 与 RM 的调用怎么缓存、怎么接进 CI 门禁、怎么监控漂移、怎么做线上线下双回路。